# Электромагнитные волны: FDTD на Python · версия 2

Три примера главы 1 Balanis: прямой перенос MATLAB-кода W. V. Andrew (1995)
на NumPy. Решаются двумерные уравнения Максвелла на сетке Йи.

**Запуск в Colab:** выберите обычную **CPU**-среду и **«Выполнить все»**.
GPU и доступ к Google Drive не нужны. Блокнот самостоятельный: исходники
решателя и тестов включены ниже, скачивать репозиторий не требуется.
Все вычисления и тесты выполняются прямо в ячейках текущего Python-ядра.
Видео появляется сразу под ячейкой расчёта.

Последовательность: определяем модель → проверяем её тестами → задаём
параметры → обновляем поля во времени → смотрим распространение волны.

| Сценарий | Модель | Возбуждение | Шаги / кадры по умолчанию |
| --- | --- | --- | --- |
| `tm_open` | Свободное пространство, PML 6 ячеек | Производная гауссиана | 111 / 37 |
| `tm_box` | Проводящая полость, отражения | Производная гауссиана | 210 / 70 |
| `te_horn` | Двумерный PEC-рупор, PML 8 ячеек | Синусоида 9,84252 ГГц с плавным включением | 210 / 70 |

Сетки — 50×50 ячеек для TMz и 100×100 для TEz. Время расчёта задаётся
`dt = dx/(2c)`; FPS меняет только скорость воспроизведения видео.
У рупора возбуждение непрерывное, а не одиночный импульс.


In [1]:
# @title 1. Подготовка окружения
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import numpy as np
import matplotlib


WORK_DIR = Path.cwd() / "balanis_fdtd"
WORK_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "Matplotlib:", matplotlib.__version__)
print(subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])
print("Рабочая папка:", WORK_DIR)


Python: 3.13.18
NumPy: 2.1.3 Matplotlib: 3.10.0
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
Рабочая папка: /content/balanis_fdtd


## 2. Решатель уравнений Максвелла

Электрические и магнитные компоненты расположены на смещённой сетке Йи.
На каждом временном шаге пространственные разности одного поля обновляют
другое. Шаг времени `dt = dx/(2c)` удовлетворяет условию устойчивости
для используемой квадратной двумерной сетки.

Ниже определён класс `Simulation`. Создание `Simulation("tm_open")`
задаёт начальные поля и поглощающую границу; `sim.step()` выполняет один
шаг расчёта. `sim.field` возвращает отображаемое поле.
Методы `_pml` и `_horn` задают поглощающие слои и проводящие стенки рупора.

Формулы, индексы и порядок обновлений сохранены из MATLAB-оригинала.
Для рупора отображается `log10(abs(Ey + 1e-6))`, для TMz — знаковое `Ez`.


In [2]:
# @title Определение решателя
class Simulation:
    """Two-dimensional vacuum TMz or TEz simulation with original defaults."""

    def __init__(self, scenario: str):
        if scenario not in ("tm_box", "tm_open", "te_horn"):
            raise ValueError(f"Unknown scenario: {scenario}")
        self.scenario = scenario
        self.te = scenario == "te_horn"
        self.size = s = 100 if self.te else 50
        self.default_steps = 210 if scenario != "tm_open" else 111
        self.npml = 8 if self.te else (6 if scenario == "tm_open" else 0)
        self.mu = 4 * np.pi * 1e-7
        self.eps = 8.854e-12
        self.c = 1 / np.sqrt(self.mu * self.eps)
        self.impedance = np.sqrt(self.mu / self.eps)
        self.frequency = 9.84252e9
        self.dx = self.c / self.frequency / 12 if self.te else 0.003
        self.dt = self.dx / self.c / 2
        self.n = 0
        if self.te:
            shapes = {"ex": (s, s + 1), "ey": (s + 1, s),
                      "hz": (s, s), "hzx": (s, s), "hzy": (s, s)}
            self.ic, self.jc = s // 2 - 21, s // 2 - 1
            components = ("ex", "ey", "hzx", "hzy")
        else:
            shapes = {"ez": (s + 1, s + 1), "ezx": (s + 1, s + 1),
                      "ezy": (s + 1, s + 1), "hx": (s + 1, s), "hy": (s, s + 1)}
            self.ic = self.jc = s // 2
            components = ("ezx", "ezy", "hx", "hy")
        self.fields = {name: np.zeros(shape) for name, shape in shapes.items()}
        self.coefficients = {}
        for name in components:
            constant = self.eps if name.startswith("e") else self.mu
            self.coefficients[name] = (np.ones(shapes[name]),
                                       np.full(shapes[name], self.dt / constant / self.dx))
        if self.npml:
            self._pml()
        if self.te:
            self._horn()

    def _pml(self):
        p, s = self.npml, self.size
        m = np.arange(1, p + 1)
        sigmax = -3 * self.eps * self.c * np.log(1e-5) / (2 * self.dx * p)
        sigma = sigmax * ((m - 0.5) / (p + 0.5)) ** 2
        rho = sigmax * self.impedance**2 * (m / (p + 0.5)) ** 2
        ca = np.exp(-sigma * self.dt / self.eps)
        cb = -(np.exp(-sigma * self.dt / self.eps) - 1) / sigma / self.dx
        da = np.exp(-rho * self.dt / self.mu)
        db = -(np.exp(-rho * self.dt / self.mu) - 1) / rho / self.dx
        # Each tuple gives the array axis normal to the absorbing boundary.
        electric = (("ey", 0), ("ex", 1)) if self.te else (("ezx", 0), ("ezy", 1))
        magnetic = (("hzx", 0), ("hzy", 1)) if self.te else (("hy", 0), ("hx", 1))
        for components, a, b, offset in ((electric, ca, cb, 1), (magnetic, da, db, 0)):
            for name, axis in components:
                aa, bb = self.coefficients[name]
                for k in range(p):
                    for position, coefficient_index in ((offset + k, p - 1 - k), (s - p + k, k)):
                        index = [slice(1, s), slice(1, s)]
                        index[axis] = position
                        aa[tuple(index)] = a[coefficient_index]
                        bb[tuple(index)] = b[coefficient_index]

    def _horn(self):
        i, j = self.ic, self.jc
        aex, bex = self.coefficients["ex"]
        aey, bey = self.coefficients["ey"]
        for wall in (j - 2, j + 2):
            aex[i - 13:i + 3, wall] = -1
            bex[i - 13:i + 3, wall] = 0
        aey[i - 13, j - 2:j + 2] = -1
        bey[i - 13, j - 2:j + 2] = 0
        for offset in range(3, 12):
            width = (offset + 3) // 2
            for wall in (j - width, j + width):
                aex[i + offset, wall] = -1
                bex[i + offset, wall] = 0
            if offset % 2:
                for wall in (j - width, j + width - 1):
                    aey[i + offset, wall] = -1
                    bey[i + offset, wall] = 0

    @property
    def field(self):
        """Signed field displayed by the MATLAB example (Ey or Ez)."""
        return self.fields["ey" if self.te else "ez"]

    def step(self):
        self.n += 1
        f, q, s = self.fields, self.coefficients, self.size
        if self.te:
            a, b = q["ex"]
            f["ex"][:, 1:s] = a[:, 1:s] * f["ex"][:, 1:s] + b[:, 1:s] * (f["hz"][:, 1:] - f["hz"][:, :-1])
            a, b = q["ey"]
            f["ey"][1:s, :] = a[1:s, :] * f["ey"][1:s, :] + b[1:s, :] * (f["hz"][:-1, :] - f["hz"][1:, :])
            f["ey"][self.ic - 10, self.jc - 2:self.jc + 2] = (
                (1 - np.exp(-(self.n / 20)**2)) * self.impedance
                * np.sin(2 * np.pi * self.frequency * self.n * self.dt))
            a, b = q["hzx"]
            f["hzx"][:] = a * f["hzx"] + b * (f["ey"][:-1, :] - f["ey"][1:, :])
            a, b = q["hzy"]
            f["hzy"][:] = a * f["hzy"] + b * (f["ex"][:, 1:] - f["ex"][:, :-1])
            f["hz"][:] = f["hzx"] + f["hzy"]
        else:
            xdiff = f["hy"][1:, 1:s] - f["hy"][:-1, 1:s]
            ydiff = f["hx"][1:s, :-1] - f["hx"][1:s, 1:]
            interior = np.s_[1:s, 1:s]
            if self.npml:
                for name, difference in (("ezx", xdiff), ("ezy", ydiff)):
                    a, b = q[name]
                    f[name][interior] = a[interior] * f[name][interior] + b[interior] * difference
                f["ez"][interior] = f["ezx"][interior] + f["ezy"][interior]
            else:
                f["ez"][interior] += self.dt / self.eps / self.dx * (xdiff + ydiff)
            dum = 4 / 25 / self.dt * (self.n * self.dt - 25 * self.dt)
            f["ez"][self.ic, self.jc] = 2 * dum * np.exp(-dum**2) / self.dx
            a, b = q["hx"]
            f["hx"][:] = a * f["hx"] + b * (f["ez"][:, :-1] - f["ez"][:, 1:])
            a, b = q["hy"]
            f["hy"][:] = a * f["hy"] + b * (f["ez"][1:, :] - f["ez"][:-1, :])
        return self.field

    def display_field(self):
        if self.te:
            # Preserve abs(Ey + epsilon), rather than silently changing the original.
            with np.errstate(divide="ignore"):
                return np.log10(np.abs(self.field + 1e-6))
        return self.field

    def save(self, path):
        """Save final staggered fields and metadata for numerical comparisons."""
        np.savez_compressed(path, **self.fields, step=self.n, dx=self.dx, dt=self.dt,
                            scenario=self.scenario, npml=self.npml)


## 3. Отображение поля

Две небольшие функции ниже создают цветовую карту и обновляют её для
очередного кадра. Сам расчёт полей находится отдельно — в `sim.step()`.
Строки массива направлены по вертикали, столбцы — по горизонтали,
как в исходном MATLAB `pcolor`. Цветовая шкала фиксирована во времени.


In [3]:
# @title Функции визуализации
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from IPython.display import Markdown, Video, display

def create_field_plot(sim):
    fig, ax = plt.subplots(figsize=(8, 7), layout="constrained")
    limits = (-6, 3) if sim.te else (-50, 50)
    rows, cols = sim.field.shape
    mesh = ax.pcolormesh(
        np.arange(1, cols + 1), np.arange(1, rows + 1), sim.display_field(),
        shading="gouraud", cmap="jet", vmin=limits[0], vmax=limits[1],
    )
    ax.set(xlim=(0, sim.size), ylim=(0, sim.size), aspect="equal",
           xlabel="j (cell index, MATLAB convention)",
           ylabel="i (cell index, MATLAB convention)")
    fig.colorbar(mesh, ax=ax, label="log10(|Ey + 10⁻⁶|)" if sim.te else "Ez (V/m)")
    title = ax.set_title("")
    return fig, mesh, title

def update_field_plot(sim, mesh, title):
    mesh.set_array(sim.display_field())
    title.set_text(
        f"{sim.scenario.upper()} · step {sim.n} · t = {sim.n * sim.dt * 1e9:.3f} ns"
    )


## 4. Численные тесты

Проверяем знаки и индексы уравнений независимыми циклами по ячейкам,
геометрию стенок, симметрию импульса и поглощение PML.
Тесты вызывают тот же класс `Simulation`, который определён выше.


In [4]:
import unittest

class FDTDTests(unittest.TestCase):
    def test_scalar_tm_updates(self):
        # Independent scalar loops catch shifted slices, signs and update ordering.
        rng = np.random.default_rng(42)
        for scenario in ("tm_box", "tm_open"):
            sim = Simulation(scenario)
            for array in sim.fields.values():
                array[:] = rng.normal(size=array.shape)
            old = {k: v.copy() for k, v in sim.fields.items()}
            expected = {k: v.copy() for k, v in old.items()}
            s = sim.size
            for i in range(1, s):
                for j in range(1, s):
                    xd = old["hy"][i, j] - old["hy"][i - 1, j]
                    yd = old["hx"][i, j - 1] - old["hx"][i, j]
                    if sim.npml:
                        for name, diff in (("ezx", xd), ("ezy", yd)):
                            a, b = sim.coefficients[name]
                            expected[name][i, j] = a[i, j] * old[name][i, j] + b[i, j] * diff
                        expected["ez"][i, j] = expected["ezx"][i, j] + expected["ezy"][i, j]
                    else:
                        expected["ez"][i, j] += sim.dt / sim.eps / sim.dx * (xd + yd)
            u = 4 / 25 / sim.dt * (sim.dt - 25 * sim.dt)
            expected["ez"][sim.ic, sim.jc] = 2 * u * np.exp(-u*u) / sim.dx
            for i in range(s + 1):
                for j in range(s):
                    a, b = sim.coefficients["hx"]
                    expected["hx"][i, j] = a[i, j]*old["hx"][i, j] + b[i, j]*(expected["ez"][i, j]-expected["ez"][i, j+1])
            for i in range(s):
                for j in range(s + 1):
                    a, b = sim.coefficients["hy"]
                    expected["hy"][i, j] = a[i, j]*old["hy"][i, j] + b[i, j]*(expected["ez"][i+1, j]-expected["ez"][i, j])
            sim.step()
            for name in expected:
                np.testing.assert_allclose(sim.fields[name], expected[name], rtol=1e-14, atol=1e-14)

    def test_scalar_te_updates(self):
        sim = Simulation("te_horn")
        rng = np.random.default_rng(43)
        for array in sim.fields.values():
            array[:] = rng.normal(size=array.shape)
        old = {k: v.copy() for k, v in sim.fields.items()}
        expected = {k: v.copy() for k, v in old.items()}
        s = sim.size
        for i in range(s):
            for j in range(1, s):
                a, b = sim.coefficients["ex"]
                expected["ex"][i, j] = a[i, j]*old["ex"][i, j] + b[i, j]*(old["hz"][i, j]-old["hz"][i, j-1])
        for i in range(1, s):
            for j in range(s):
                a, b = sim.coefficients["ey"]
                expected["ey"][i, j] = a[i, j]*old["ey"][i, j] + b[i, j]*(old["hz"][i-1, j]-old["hz"][i, j])
        source = (1-np.exp(-(1/20)**2))*sim.impedance*np.sin(2*np.pi*sim.frequency*sim.dt)
        expected["ey"][sim.ic-10, sim.jc-2:sim.jc+2] = source
        for i in range(s):
            for j in range(s):
                for name, diff in (("hzx", expected["ey"][i, j]-expected["ey"][i+1, j]),
                                   ("hzy", expected["ex"][i, j+1]-expected["ex"][i, j])):
                    a, b = sim.coefficients[name]
                    expected[name][i, j] = a[i, j]*old[name][i, j]+b[i, j]*diff
                expected["hz"][i, j] = expected["hzx"][i, j]+expected["hzy"][i, j]
        sim.step()
        for name in expected:
            np.testing.assert_allclose(sim.fields[name], expected[name], rtol=1e-14, atol=1e-14)

    def test_original_geometry_and_pml_indices(self):
        sim = Simulation("te_horn")
        i, j = sim.ic, sim.jc
        # Explicit MATLAB (one-based) wall locations, converted independently.
        ex_walls = {(ii-1, jj-1) for ii in range(17, 33) for jj in (48, 52)}
        for ii, lo, hi in ((33,47,53),(34,47,53),(35,46,54),(36,46,54),
                           (37,45,55),(38,45,55),(39,44,56),(40,44,56),(41,43,57)):
            ex_walls.update(((ii-1, lo-1), (ii-1, hi-1)))
        ey_walls = {(16, jj-1) for jj in range(48,52)}
        for ii, lo, hi in ((33,47,52),(35,46,53),(37,45,54),(39,44,55),(41,43,56)):
            ey_walls.update(((ii-1, lo-1), (ii-1, hi-1)))
        for name, walls in (("ex",ex_walls),("ey",ey_walls)):
            a, b = sim.coefficients[name]
            self.assertEqual(set(map(tuple, np.argwhere(b == 0))), walls)
            self.assertTrue(all(a[index] == -1 for index in walls))
        for scenario in ("tm_open", "te_horn"):
            sim = Simulation(scenario)
            p, s = sim.npml, sim.size
            sigmax = -3*sim.eps*sim.c*np.log(1e-5)/(2*sim.dx*p)
            for name, axis, magnetic in (("ey" if sim.te else "ezx",0,False),
                                         ("ex" if sim.te else "ezy",1,False),
                                         ("hzx" if sim.te else "hy",0,True),
                                         ("hzy" if sim.te else "hx",1,True)):
                a, b = sim.coefficients[name]
                for k in range(1,p+1):
                    loss = (sigmax*sim.impedance**2*(k/(p+.5))**2 if magnetic
                            else sigmax*((k-.5)/(p+.5))**2)
                    constant = sim.mu if magnetic else sim.eps
                    aa = np.exp(-loss*sim.dt/constant)
                    bb = -(aa-1)/loss/sim.dx
                    for position in ((p-k if magnetic else p+1-k), s-p+k-1):
                        index = (position,s//2) if axis == 0 else (s//2,position)
                        self.assertAlmostEqual(a[index], aa)
                        self.assertAlmostEqual(b[index], bb)

    def test_symmetry_pec_and_absorption(self):
        box, opened = Simulation("tm_box"), Simulation("tm_open")
        for _ in range(210):
            box.step()
            opened.step()
            np.testing.assert_allclose(box.field, box.field.T, atol=1e-10)
            np.testing.assert_allclose(box.field, box.field[::-1], atol=1e-10)
        for sim in (box, opened):
            self.assertTrue(np.isfinite(sim.field).all())
            self.assertTrue(np.all(sim.field[[0,-1],:] == 0))
            self.assertTrue(np.all(sim.field[:,[0,-1]] == 0))
        # Compare the physical interior after the outgoing pulse has left it.
        core = np.s_[8:-8,8:-8]
        self.assertLess(np.linalg.norm(opened.field[core]), .1*np.linalg.norm(box.field[core]))
        horn = Simulation("te_horn")
        for _ in range(210):
            horn.step()
        for name in ("ex", "ey"):
            walls = horn.coefficients[name][1] == 0
            self.assertTrue(np.all(horn.fields[name][walls] == 0))
        self.assertTrue(all(np.isfinite(a).all() for a in horn.fields.values()))


In [5]:
# @title Запуск тестов прямо в ядре блокнота
suite = unittest.defaultTestLoader.loadTestsFromTestCase(FDTDTests)
test_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Численные тесты не прошли — проверьте решатель выше")


test_original_geometry_and_pml_indices (__main__.FDTDTests.test_original_geometry_and_pml_indices) ... ok
test_scalar_te_updates (__main__.FDTDTests.test_scalar_te_updates) ... ok
test_scalar_tm_updates (__main__.FDTDTests.test_scalar_tm_updates) ... ok
test_symmetry_pec_and_absorption (__main__.FDTDTests.test_symmetry_pec_and_absorption) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.268s

OK


In [7]:
# @title 5. Параметры моделирования
SCENARIO = "tm_open" # @param ["all", "tm_open", "tm_box", "te_horn"]
STEPS = 0 # @param {type:"integer"}
FRAME_STRIDE = 3 # @param {type:"integer"}
FPS = 7 # @param {type:"integer"}

# STEPS = 0: исходная длительность каждого сценария.
# При изменении параметров повторите выполнение этой и следующих ячеек.
DEFAULT_STEPS = {"tm_open": 111, "tm_box": 210, "te_horn": 210}
if SCENARIO not in ("all", *DEFAULT_STEPS):
    raise ValueError("Неизвестный сценарий")
for name, value in (("STEPS", STEPS), ("FRAME_STRIDE", FRAME_STRIDE), ("FPS", FPS)):
    if not isinstance(value, int) or isinstance(value, bool):
        raise ValueError(f"{name} должен быть целым числом")
if STEPS < 0 or FRAME_STRIDE < 1 or FPS < 1:
    raise ValueError("STEPS должен быть >= 0, FRAME_STRIDE и FPS — >= 1")
scenarios = list(DEFAULT_STEPS) if SCENARIO == "all" else [SCENARIO]
if any((STEPS or DEFAULT_STEPS[name]) < FRAME_STRIDE for name in scenarios):
    raise ValueError("Число шагов должно быть не меньше интервала между кадрами")
print("Будут рассчитаны:", ", ".join(scenarios))


Будут рассчитаны: tm_open


## 6. Расчёт и видео

В следующей ячейке создаётся `sim = Simulation(name)`. Временной цикл
вызывает `sim.step()` и снимает кадр каждые `FRAME_STRIDE` шагов.
`FFMpegWriter` кодирует кадры в MP4, а `Video(..., embed=True)` вставляет
видео в вывод этой же ячейки.

MP4, финальный снимок PNG и поля NPZ сохраняются для скачивания.
Можно изменить параметры выше и повторить расчёт: каждый сценарий
начинается с новых нулевых полей. Вывод ячейки заменится новым видео.


In [8]:
# @title Распространение волны и видео
import time
generated = []
for name in scenarios:
    start = time.perf_counter()
    sim = Simulation(name)
    steps = STEPS or sim.default_steps
    video_path = OUTPUT_DIR / f"{name}.mp4"
    fig, mesh, title = create_field_plot(sim)
    writer = FFMpegWriter(fps=FPS, codec="libx264", extra_args=["-pix_fmt", "yuv420p"])
    try:
        with writer.saving(fig, str(video_path), dpi=100):
            for step in range(steps):
                sim.step()  # Один шаг решения уравнений Максвелла.
                if sim.n % FRAME_STRIDE == 0:
                    update_field_plot(sim, mesh, title)
                    writer.grab_frame()
        update_field_plot(sim, mesh, title)
        fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=150)
        sim.save(OUTPUT_DIR / f"{name}.npz")
    finally:
        plt.close(fig)
    generated.append(name)
    print(f"{name}: {steps} шагов, {steps // FRAME_STRIDE} кадров, "
          f"{time.perf_counter() - start:.1f} с", flush=True)
    display(Markdown(f"### {name}"))
    display(Video(str(video_path), embed=True, mimetype="video/mp4", width=720,
                  html_attributes="controls loop playsinline"))


tm_open: 111 шагов, 37 кадров, 9.5 с


### tm_open

In [ ]:
# @title 7. Проверка видео и сохранённых полей
import json
from fractions import Fraction
report = []
for name in generated:
    video = OUTPUT_DIR / f"{name}.mp4"
    probe = json.loads(subprocess.check_output([
        "ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
        "-show_entries", "stream=codec_name,width,height,r_frame_rate,nb_read_frames",
        "-of", "json", str(video),
    ], text=True))["streams"][0]
    expected_steps = STEPS or DEFAULT_STEPS[name]
    assert probe["codec_name"] == "h264", probe
    assert int(probe["nb_read_frames"]) == expected_steps // FRAME_STRIDE, probe
    assert Fraction(probe["r_frame_rate"]) == FPS, probe
    assert (probe["width"], probe["height"]) == (800, 700), probe
    with np.load(OUTPUT_DIR / f"{name}.npz") as fields:
        assert int(fields["step"]) == expected_steps
        assert str(fields["scenario"]) == name
        for key in (("ex", "ey", "hz") if name == "te_horn" else ("ez", "hx", "hy")):
            assert np.isfinite(fields[key]).all(), (name, key)
        field = fields["ey" if name == "te_horn" else "ez"]
        maximum = float(np.max(np.abs(field)))
    report.append(dict(scenario=name, steps=expected_steps,
                       frames=int(probe["nb_read_frames"]), fps=FPS,
                       maximum_field=maximum, video_bytes=video.stat().st_size))
    print(f"OK: {name}, {probe['nb_read_frames']} кадров, {FPS} fps, max |field|={maximum:.8g}")
_ = (OUTPUT_DIR / "verification.json").write_text(json.dumps(report, indent=2))


Видео встроены в вывод ячеек. Финальные PNG и численные массивы NPZ находятся
в `balanis_fdtd/output`; их также можно скачать через панель «Файлы» Colab.
Финальный снимок `tm_open` почти однородный: к концу расчёта волна уже уходит
в поглощающую границу — процесс виден на видео.

Следующая ячейка создаёт ZIP текущего запуска. Чтобы скачать его на компьютер,
включите `DOWNLOAD_RESULTS` и выполните эту ячейку. Файлы на VM временные:
сохраните нужные результаты перед завершением сессии.


In [ ]:
# @title 8. Архив результатов
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
import zipfile
archive = WORK_DIR / "fdtd_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for name in generated:
        for suffix in ("mp4", "png", "npz"):
            path = OUTPUT_DIR / f"{name}.{suffix}"
            z.write(path, arcname=path.name)
    z.write(OUTPUT_DIR / "verification.json", arcname="verification.json")
print(f"Архив: {archive} ({archive.stat().st_size / 1024:.0f} КиБ)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))
print("FDTD_NOTEBOOK_OK")
